In [0]:
file_list = dbutils.widgets.get("file_list")
cashrecpts_path = dbutils.widgets.get("cashrecpts_path")
landing_table = dbutils.widgets.get("landing_table")

In [0]:
import ast
file_list = ast.literal_eval(file_list)
for file_name in file_list:
  file_path = cashrecpts_path + file_name
  df_cashrecpts = spark.read.option("header", "true").option("delimiter", "\t").csv(file_path)
  df_cashrecpts.createOrReplaceTempView("cashrecpts_view")

  spark.sql(f"""
  INSERT INTO {landing_table}
  SELECT
    `PostedDt` AS posted_dt,
    `Payer` AS payer,
    `BillTo#` AS bill_to_num,
    `BillToName` AS bill_to_name,
    `Program` AS program,
    `Team` AS team,
    `EntryDt` AS entry_dt,
    `Client#` AS client_num,
    `Invoice#` AS invoice_num,
    `BillDate` AS bill_date,
    `WeDate` AS we_date,
    `Product` AS product,
    `Applied` AS applied,
    `AppliedOnMR` AS applied_on_mr,
    `DepositDt` AS deposit_dt,
    `BatchID` AS batch_id,
    `CheckID` AS check_id,
    `Payment Number` AS payment_number,
    `Ofc` AS ofc,
    `Company` AS company,
    `Type` AS type,
    `Batch#?` AS batch_num,
    `Tr#` AS tr_num,
    `Bank` AS bank,
    current_timestamp() AS _load_timestamp,
    "{file_name[:-4]}" AS _file_name
  FROM cashrecpts_view
  """)